# Step 4 — Historical Analysis: Outage x Weather

**Objective:** Merge panels, compute outage rates by compound event category, run two-way FE panel regressions, logistic regression, and RD.

**Model:**
OutageSeverity_ct = beta1*HeatwaveDay + beta2*CompoundHeatWind + beta3*CompoundTriple
                  + gamma*X_ct + alpha_c + delta_t + epsilon_ct

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import PROCESSED, RD_THRESHOLD_ERCOT_C, RD_THRESHOLD_CAISO_C
from src.analysis.compound_flags import outage_rates_by_category
from src.analysis.panel_regression import run_panel_ols, run_logit, summarise_results
from src.analysis.rd_analysis import run_rd, rd_bandwidth_sensitivity
from src.viz.maps import outage_heatmap, rd_plot, outage_rate_bar


## 4.1 Load processed panels

In [ ]:
def load_panel(path):
    if path.exists():
        df = pd.read_csv(path, parse_dates=['date'])
        df['fips'] = df['fips'].astype(str).str.zfill(5)
        return df.set_index(['fips', 'date'])
    print(f'Not found: {path}')
    return None

outage_ercot  = load_panel(PROCESSED['outage_ercot'])
weather_ercot = load_panel(PROCESSED['weather_ercot'])
outage_caiso  = load_panel(PROCESSED['outage_caiso'])
weather_caiso = load_panel(PROCESSED['weather_caiso'])
print('ERCOT outage:', outage_ercot.shape  if outage_ercot  is not None else 'missing')
print('ERCOT weather:', weather_ercot.shape if weather_ercot is not None else 'missing')


## 4.2 Merge panels

In [ ]:
merged_ercot = merged_caiso = None
if outage_ercot is not None and weather_ercot is not None:
    merged_ercot = outage_ercot.join(weather_ercot, how='inner')
    print(f'Merged ERCOT panel: {merged_ercot.shape}')
if outage_caiso is not None and weather_caiso is not None:
    merged_caiso = outage_caiso.join(weather_caiso, how='inner')
    print(f'Merged CAISO panel: {merged_caiso.shape}')


## 4.3 Outage rates by weather category

Replicates the key finding from the Nature 2025 EAGLE-I study: compound triple events (heat + wind + precip) amplify outage probability by ~52x.

In [ ]:
if merged_ercot is not None:
    rates = outage_rates_by_category(merged_ercot.reset_index())
    print('ERCOT outage rates by category:')
    print(rates.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    outage_rate_bar(rates, ax=ax, title='ERCOT: Outage rates by compound event category')
    plt.tight_layout()
    plt.savefig('../data/processed/ercot_outage_rates.png', dpi=150)
    plt.show()


## 4.4 Outage severity heatmap (county x time)

In [ ]:
if merged_ercot is not None:
    fig, ax = plt.subplots(figsize=(16, 9))
    outage_heatmap(
        merged_ercot.reset_index(),
        value_col='outage_fraction',
        top_n_counties=30,
        title='ERCOT outage severity — top 30 most-affected counties',
        ax=ax,
    )
    plt.tight_layout()
    plt.savefig('../data/processed/ercot_outage_heatmap.png', dpi=150)
    plt.show()


## 4.5 Fixed-effects panel OLS

In [ ]:
if merged_ercot is not None:
    result_ols = run_panel_ols(
        merged_ercot,
        outcome='total_customer_hours',
        weather_vars=['heatwave_day', 'compound_heat_wind', 'compound_triple'],
    )
    print(result_ols.summary)
    print('\nCoefficient table:')
    print(summarise_results(result_ols))


## 4.6 Logistic regression for binary outage events

In [ ]:
if merged_ercot is not None:
    result_logit = run_logit(
        merged_ercot,
        outcome='outage_event_flag',
        weather_vars=['heatwave_day', 'compound_heat_wind', 'compound_triple'],
    )
    print(result_logit.summary())


## 4.7 Regression Discontinuity — ERCOT heat alert threshold (36°C)

In [ ]:
if merged_ercot is not None:
    rd_result = run_rd(
        merged_ercot.reset_index(),
        cutoff=RD_THRESHOLD_ERCOT_C,
        running_var='tmax',
        outcome='total_customer_hours',
    )
    print(rd_result.summary())

    bw_sens = rd_bandwidth_sensitivity(merged_ercot.reset_index(), cutoff=RD_THRESHOLD_ERCOT_C)
    print('\nBandwidth sensitivity:')
    print(bw_sens)

    fig, ax = plt.subplots(figsize=(10, 6))
    rd_plot(merged_ercot.reset_index(), rd_result, ax=ax)
    plt.tight_layout()
    plt.savefig('../data/processed/ercot_rd_plot.png', dpi=150)
    plt.show()


## 4.8 Save merged panels

In [ ]:
if merged_ercot is not None:
    merged_ercot.to_csv(PROCESSED['merged_ercot'])
    print('Saved:', PROCESSED['merged_ercot'])
if merged_caiso is not None:
    merged_caiso.to_csv(PROCESSED['merged_caiso'])
    print('Saved:', PROCESSED['merged_caiso'])
